<a href="https://colab.research.google.com/github/gauravd12345/language_models/blob/main/seq2seq/seq2seq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split

import nltk
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize

dataset = "hf://datasets/PaulineSanchez/Translation_words_and_sentences_english_french/data/train-00000-of-00001-3d14582ea46e1b17.parquet"
df = pd.read_parquet(dataset)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

In [ ]:
c1, c2 = df.columns
enc = np.array(df[c1])
dec = np.array(df[c2])

for i in range(len(enc)):
  enc[i] = enc[i].lower()
  dec[i] = dec[i].lower()

for i in np.random.choice(np.arange(len(df)), 10):
  print(f"{enc[i]:<50} | {dec[i]}")

In [ ]:
""" setup for encoder vocabulary """
enc_words = [word_tokenize(seq) for seq in enc]
enc_vocab = set() # set of unique words
for seq in enc_words:
  for word in seq:
    enc_vocab.add(word)

enc_vocab.add("<pad>")
enc_vocab.add("<sos>")
enc_vocab.add("<eos>")
enc_vocab = sorted(enc_vocab)
enc_vocab_len = len(enc_vocab)

etoi, itoe = {}, {}
for i in range(enc_vocab_len):
    etoi[enc_vocab[i]] = i
    itoe[i] = enc_vocab[i]

""" setup for decoder vocabulary """
dec_words = [word_tokenize(seq) for seq in dec]
dec_vocab = set()
for seq in dec_words:
  for word in seq:
    dec_vocab.add(word)

dec_vocab.add("<pad>")
dec_vocab.add("<sos>")
dec_vocab.add("<eos>")
dec_vocab = sorted(dec_vocab)
dec_vocab_len = len(dec_vocab)

dtoi, itod = {}, {}
for i in range(dec_vocab_len):
    dtoi[dec_vocab[i]] = i
    itod[i] = dec_vocab[i]

In [ ]:
print(f"english vocab size: {enc_vocab_len}")
print(np.random.choice(enc_vocab, 10))

print(f"\nfrench vocab size: {dec_vocab_len}")
print(np.random.choice(dec_vocab, 10))

In [ ]:
""" Hyperparameters """

embedding_dim = 300
hidden_size = 300 # context vector length
max_decoder_seq = 50
batch_size = 128

lr = 0.001
epochs = 7

enc_pad_tok = etoi["<pad>"]
enc_sos_tok = etoi["<sos>"]
enc_eos_tok = etoi["<eos>"]

dec_pad_tok = etoi["<pad>"]
dec_sos_tok = dtoi["<sos>"]
dec_eos_tok = dtoi["<eos>"]

In [ ]:
E = []
for sentence in enc_words:
  seq = []
  tok = ["<sos>"] + sentence + ["<eos>"]
  for word in tok:
    e_i = etoi[word]
    seq.append(e_i)
  E.append(torch.tensor(seq).to(device))

D = []
for sentence in dec_words:
  seq = []
  tok = ["<sos>"] + sentence + ["<eos>"]
  for word in tok:
    d_i = dtoi[word]
    seq.append(d_i)
  D.append(torch.tensor(seq).to(device))

In [ ]:
print(f"No. of samples: {len(E)}")
for i in np.random.choice(np.arange(len(E)), 10):
    english = " ".join(itoe[etok.item()] for etok in E[i])
    french  = " ".join(itod[dtok.item()] for dtok in D[i])
    print(f"[{i:2}] EN: {english:<50} | FR: {french}")

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc_embed = nn.Embedding(enc_vocab_len, embedding_dim)
        self.dec_embed = nn.Embedding(dec_vocab_len, embedding_dim)

        """ encoder RNN """
        self.E_x = nn.Linear(embedding_dim, hidden_size)
        self.E_h = nn.Linear(hidden_size, hidden_size)
        self.E_y = nn.Linear(hidden_size, hidden_size)

        """ decoder RNN """
        self.D_x = nn.Linear(embedding_dim, hidden_size)
        self.D_h = nn.Linear(hidden_size, hidden_size)
        self.D_y = nn.Linear(hidden_size, dec_vocab_len)

    def forward(self, e, d=None):
        s = torch.zeros(e.size(0), hidden_size).to(device)  # initial hidden state s(0)
        w = self.enc_embed(e)
        for i in range(e.size(1)):
          w_i = w[:, i, :]
          s = torch.tanh(self.E_x(w_i) + self.E_h(s))
          y = self.E_y(s)

        out = []
        if d is not None: # training mode(teacher forcing)
          w = self.dec_embed(d)
          for i in range(d.size(1)):
            w_i = w[:, i, :]
            s = torch.tanh(self.D_x(w_i) + self.D_h(s))
            y = self.D_y(s)
            out.append(y)

        else:
          start = torch.tensor(dec_sos_tok).to(device)
          w = self.dec_embed(start)
          for _ in range(max_decoder_seq):
            s = torch.tanh(self.D_x(w) + self.D_h(s))
            y = self.D_y(s)
            out.append(y)
            pred = torch.argmax(y, dim=1).item()

            w = self.dec_embed(torch.tensor(pred).to(device))

        return torch.stack(out, dim=1), s

seq2seq = Seq2Seq().to(device)

In [ ]:
class Seq2SeqDataset(Dataset):
  def __init__(self, e, d):
    self.e = e
    self.d = d

  def __getitem__(self, index):
     return self.e[index], self.d[index]

  def __len__(self):
    return len(self.e)

""" collate function for batching """
def collate_fn(batch):
  e, d = zip(*batch)

  max_e = max(len(seq) for seq in e)
  max_d = max(len(seq) for seq in d)

  pad_e = torch.zeros(len(e), max_e, dtype=torch.long).to(device)
  pad_d = torch.zeros(len(d), max_d, dtype=torch.long).to(device)

  for i in range(len(e)):
    pad_e[i] = torch.cat([e[i], torch.tensor([enc_pad_tok] * (max_e - len(e[i]))).to(device)])

  for i in range(len(d)):
    pad_d[i] = torch.cat([d[i], torch.tensor([dec_pad_tok] * (max_d - len(d[i]))).to(device)])

  return pad_e, pad_d

In [ ]:
data_len = 30000
dataset = Seq2SeqDataset(E[:data_len], D[:data_len])
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)

criterion = nn.CrossEntropyLoss(ignore_index=dec_pad_tok)
optimizer = optim.Adam(seq2seq.parameters(), lr=lr)

for epoch in range(epochs):
  total_loss = 0.0
  for e, d in dataloader:
    e = e.to(device)
    d = d.to(device)

    optimizer.zero_grad()

    d_in, d_out = d[:, :-1], d[:, 1:]
    out, _ = seq2seq(e[:, :-1], d_in)

    loss = criterion(
        out.reshape(-1, out.size(-1)),
        d_out.reshape(-1)
    )

    total_loss += loss.item()

    loss.backward()
    optimizer.step()

  print(f"Epoch: {epoch+1}/{epochs} | loss: {total_loss / len(dataloader)}")

In [ ]:
test = "What is your name"
seq = []
tok =  ["<sos>"] + word_tokenize(test.lower()) + ["<eos>"]
for word in tok:
    e_i = etoi[word]
    seq.append(e_i)

seq2seq.eval()
with torch.no_grad():
  seq = torch.tensor(seq).to(device)
  seq = seq.unsqueeze(0)

  pred, _ = seq2seq(seq)
  for prob in pred[0]:
      word = itod[torch.argmax(prob).item()]
      if word == "<eos>":
          break

      print(word, end=" ")

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc_embed = nn.Embedding(enc_vocab_len, embedding_dim)
        self.dec_embed = nn.Embedding(dec_vocab_len, embedding_dim)

        """ encoder/decoder RNN """
        self.encoder = nn.RNN(embedding_dim, hidden_size, batch_first=True)
        self.decoder = nn.RNN(embedding_dim, hidden_size, batch_first=True)

        self.fc = nn.Linear(hidden_size, dec_vocab_len)

    def forward(self, e, d=None):
        s = torch.zeros(1, e.size(0), hidden_size).to(device)  # initial hidden state s(0)
        w = self.enc_embed(e)
        _, s = self.encoder(w, s) # context vector

        w = self.dec_embed(d)
        out, _ = self.decoder(w, s)

        out = self.fc(out)

        return out

seq2seq = Seq2Seq().to(device)

In [ ]:
dataset = Seq2SeqDataset(E, D)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)

criterion = nn.CrossEntropyLoss(ignore_index=dec_pad_tok)
optimizer = optim.Adam(seq2seq.parameters(), lr=lr)

for epoch in range(epochs):
  total_loss = 0.0
  for e, d in dataloader:
    e = e.to(device)
    d = d.to(device)

    optimizer.zero_grad()

    d_in, d_out = d[:, :-1], d[:, 1:]
    out = seq2seq(e[:, :-1], d_in)

    loss = criterion(
        out.reshape(-1, out.size(-1)),
        d_out.reshape(-1)
    )

    total_loss += loss.item()

    loss.backward()
    optimizer.step()

  print(f"Epoch: {epoch+1}/{epochs} | loss: {total_loss / len(dataloader)}")